<a href="https://colab.research.google.com/github/Kareena-3/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Baseline Score

**Lane:** CTR / Engagement Opportunity Scoring  
**Notebook:** `work/notebooks/w04_baseline_score.ipynb`

This notebook continues the submitted Week 3 contract without changing it: one client-content item, February 2026 as the feature window, March 2026 as the outcome window, and the existing March `went_dark` outcome. The baseline is intentionally simple and uses only information available by February 28.

## 1. Setup

In [1]:
import os, getpass, duckdb, numpy as np, pandas as pd
def get_hf_token():
    return os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
con=duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [get_hf_token()])
REL="hf://datasets/FlyRank/internship-warehouse"
FACT=f"{REL}/fact_content_daily_performance"
FEB=f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR=f"read_parquet('{FACT}/month=2026-03/*.parquet')"
print("Connected: February features -> March outcome")

Paste your Hugging Face READ token (hf_...): ··········
Connected: February features -> March outcome


## 2. Recreate the Week 3 feature universe

The baseline itself only uses February observations. March is not touched until evaluation.

In [2]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW feb_agg AS
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS imp_feb,
       SUM(gsc_clicks) AS clk_feb,
       SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0) AS avg_position_feb,
       COUNT(*) FILTER (WHERE gsc_data_available) AS measured_gsc_days,
       SUM(CASE WHEN ga4_data_available THEN ga4_sessions ELSE 0 END) AS sessions_feb,
       SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_feb
FROM {FEB}
GROUP BY client_hash_id, content_hash_id
""")
frame=con.sql("""
SELECT * FROM feb_agg
WHERE measured_gsc_days > 0 AND imp_feb >= 100
""").df()
frame["ctr_feb"]=frame.clk_feb/frame.imp_feb.replace(0,np.nan)
frame["engagement_rate_feb"]=frame.engaged_sessions_feb/frame.sessions_feb.replace(0,np.nan)
print(f"Eligible February pages: {len(frame):,}")

Eligible February pages: 80,322


## 3. Position-adjusted CTR context

Raw CTR is not compared across unrelated positions. Expected CTR is calculated inside broad February position tiers.

In [3]:
def position_tier(p):
    if pd.isna(p) or p<=0: return "unknown"
    if p<=3: return "1-3"
    if p<=10: return "4-10"
    if p<=20: return "11-20"
    return "21+"
frame["position_tier_feb"]=frame.avg_position_feb.apply(position_tier)
tier=(frame.groupby("position_tier_feb")
      .agg(tier_impressions=("imp_feb","sum"),tier_clicks=("clk_feb","sum")))
tier["expected_ctr_feb"]=tier.tier_clicks/tier.tier_impressions.replace(0,np.nan)
frame=frame.merge(tier[["expected_ctr_feb"]],left_on="position_tier_feb",right_index=True,how="left")
frame["ctr_gap"]=(frame.expected_ctr_feb-frame.ctr_feb).clip(lower=0)
display(tier.reset_index())

,position_tier_feb,tier_impressions,tier_clicks,expected_ctr_feb
0,1-3,34656796.0,137099.0,0.003956
1,11-20,20741436.0,65618.0,0.003164
2,21+,20639477.0,31746.0,0.001538
3,4-10,102370754.0,345657.0,0.003377
4,unknown,605.0,0.0,0.000000


## 4. Transparent baseline score

**40% visibility at stake + 35% position-adjusted CTR gap + 25% position opportunity.** These are explicit policy weights, not learned from March.

In [4]:
def minmax(s):
    lo,hi=s.min(),s.max()
    return pd.Series(0.0,index=s.index) if pd.isna(lo) or pd.isna(hi) or lo==hi else (s-lo)/(hi-lo)
frame["visibility_component"]=minmax(np.log1p(frame.imp_feb))
frame["ctr_gap_component"]=minmax(frame.ctr_gap.fillna(0))
frame["position_component"]=np.where(
    (frame.avg_position_feb>0)&(frame.avg_position_feb<=20),
    1-((frame.avg_position_feb-1)/19),0)
frame["position_component"]=np.clip(frame.position_component,0,1)
frame["baseline_score"]=100*(.40*frame.visibility_component+.35*frame.ctr_gap_component+.25*frame.position_component)
assert frame.baseline_score.between(0,100).all()
frame[["visibility_component","ctr_gap_component","position_component","baseline_score"]].describe()

,visibility_component,ctr_gap_component,position_component,baseline_score
count,80322.000000,80322.000000,80322.000000,80322.000000
mean,0.273383,0.411264,0.579056,39.805967
std,0.176234,0.355927,0.323070,16.206174
min,0.000000,0.000000,0.000000,0.000000
25%,0.127780,0.000000,0.359373,27.082609
50%,0.254470,0.388816,0.678462,40.744070
75%,0.400307,0.799721,0.837596,52.271418
max,1.000000,1.000000,1.000000,99.913004


## 5. Reason codes

In [6]:
def reasons(r):
    x=[]
    if r.imp_feb>=frame.imp_feb.quantile(.75): x.append("high_visibility")
    if pd.notna(r.ctr_feb) and pd.notna(r.expected_ctr_feb) and r.ctr_feb<r.expected_ctr_feb: x.append("below_position_tier_ctr")
    if 0<r.avg_position_feb<=10: x.append("page_one_position")
    elif 10<r.avg_position_feb<=20: x.append("near_page_one")
    if pd.notna(r.engagement_rate_feb) and r.engagement_rate_feb<.30: x.append("weak_engagement_context")
    return "; ".join(x) if x else "monitor"
frame["reason_codes"]=frame.apply(reasons,axis=1)
queue=frame.sort_values(["baseline_score","imp_feb"],ascending=False)[[
"client_hash_id","content_hash_id","baseline_score","reason_codes","imp_feb","clk_feb","ctr_feb",
"avg_position_feb","position_tier_feb","expected_ctr_feb","ctr_gap","sessions_feb","engagement_rate_feb"]].reset_index(drop=True)
display(queue.head(20))

,client_hash_id,content_hash_id,baseline_score,reason_codes,imp_feb,clk_feb,ctr_feb,avg_position_feb,position_tier_feb,expected_ctr_feb,ctr_gap,sessions_feb,engagement_rate_feb
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,99.913004,high_visibility; below_position_tier_ctr; page...,203401.0,2.0,0.000010,0.259483,1-3,0.003956,0.003946,0.0,NaN
1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,99.750451,high_visibility; below_position_tier_ctr; page...,195648.0,1.0,0.000005,0.009865,1-3,0.003956,0.003951,0.0,NaN
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,99.749951,high_visibility; below_position_tier_ctr; page...,193954.0,0.0,0.000000,0.070336,1-3,0.003956,0.003956,0.0,NaN
3,client_73cda7b4e4f265ea,content_c9f840183215651b,95.720248,high_visibility; below_position_tier_ctr; page...,125035.0,0.0,0.000000,2.308282,1-3,0.003956,0.003956,0.0,NaN
4,client_23a62021009f63c4,content_2ac8c7995de53cd1,95.451751,high_visibility; below_position_tier_ctr; page...,92128.0,4.0,0.000043,0.039630,1-3,0.003956,0.003912,9.0,0.111111
5,client_23a62021009f63c4,content_44f34c0a90047651,94.451217,high_visibility; below_position_tier_ctr; page...,90223.0,13.0,0.000144,0.518504,1-3,0.003956,0.003812,17.0,0.000000
6,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,90.092963,high_visibility; below_position_tier_ctr; page...,51000.0,13.0,0.000255,1.287686,1-3,0.003956,0.003701,0.0,NaN
7,client_e547b89c05043229,content_306bc78dff1eb683,89.341263,high_visibility; below_position_tier_ctr; page...,83657.0,52.0,0.000622,1.370883,1-3,0.003956,0.003334,49.0,0.020408
8,client_23a62021009f63c4,content_4fe94bdfd75c38f9,87.387710,high_visibility; below_position_tier_ctr; page...,42351.0,20.0,0.000472,1.139690,1-3,0.003956,0.003484,28.0,0.035714
9,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,87.387323,high_visibility; below_position_tier_ctr; page...,18472.0,0.0,0.000000,0.626353,1-3,0.003956,0.003956,0.0,NaN


## 6. Freeze the score, then attach the March outcome

This is the key leakage check: March data is used only after the February ranking is complete.

In [7]:
march=con.sql(f"""
SELECT client_hash_id,content_hash_id,
       SUM(gsc_impressions) FILTER (WHERE gsc_data_available) AS imp_mar,
       SUM(gsc_clicks) FILTER (WHERE gsc_data_available) AS clk_mar,
       COUNT(*) FILTER (WHERE gsc_data_available) AS measured_gsc_days_mar
FROM {MAR}
GROUP BY client_hash_id,content_hash_id
""").df()
evaluation=queue.merge(march,on=["client_hash_id","content_hash_id"],how="left")
evaluation=evaluation[evaluation.measured_gsc_days_mar.fillna(0)>0].copy()
evaluation[["imp_mar","clk_mar"]]=evaluation[["imp_mar","clk_mar"]].fillna(0)
evaluation["went_dark"]=(evaluation.clk_mar==0).astype(int)
print(f"Measured March outcomes: {len(evaluation):,}")
print(f"went_dark base rate: {evaluation.went_dark.mean():.3%}")

Measured March outcomes: 76,702
went_dark base rate: 34.012%


## 7. Evaluate the baseline

For a ranked review queue, Precision@K answers: **among the top K pages sent for review, how many had the Week 3 March outcome?**

In [8]:
def precision_at_k(df,k):
    top=df.nlargest(k,"baseline_score")
    return top.went_dark.mean() if len(top) else np.nan
results=pd.DataFrame({"K":[20,50,100],
"Precision@K":[precision_at_k(evaluation,20),precision_at_k(evaluation,50),precision_at_k(evaluation,100)]})
display(results.style.format({"Precision@K":"{:.3f}"}))
print(f"Random-ranking reference rate: {evaluation.went_dark.mean():.3%}")

,K,Precision@K
0,20,0.050
1,50,0.140
2,100,0.140


Random-ranking reference rate: 34.012%


## 8. Sensitivity check

The baseline weights are policy choices. Nearby alternatives are compared to check whether the result depends entirely on one arbitrary weighting.

In [9]:
weight_sets={"main_40_35_25":(.40,.35,.25),"visibility_heavy_50_30_20":(.50,.30,.20),"ctr_heavy_30_50_20":(.30,.50,.20)}
rows=[]
for name,(a,b,c) in weight_sets.items():
    tmp=frame[["client_hash_id","content_hash_id","visibility_component","ctr_gap_component","position_component"]].copy()
    tmp["score"]=100*(a*tmp.visibility_component+b*tmp.ctr_gap_component+c*tmp.position_component)
    tmp=tmp.merge(evaluation[["client_hash_id","content_hash_id","went_dark"]],on=["client_hash_id","content_hash_id"])
    rows.append({"weight_policy":name,"Precision@20":tmp.nlargest(20,"score").went_dark.mean(),
                 "Precision@50":tmp.nlargest(50,"score").went_dark.mean(),
                 "Precision@100":tmp.nlargest(100,"score").went_dark.mean()})
display(pd.DataFrame(rows).style.format({"Precision@20":"{:.3f}","Precision@50":"{:.3f}","Precision@100":"{:.3f}"}))

,weight_policy,Precision@20,Precision@50,Precision@100
0,main_40_35_25,0.050,0.140,0.140
1,visibility_heavy_50_30_20,0.050,0.100,0.080
2,ctr_heavy_30_50_20,0.200,0.240,0.260


## 9. Final interpretation

### What I built
A transparent baseline that ranks pages using February-only visibility, position-adjusted CTR under-capture, and position opportunity.

### Why it is leakage-safe
- Score inputs stop on February 28.
- March is only attached after the ranking is frozen.
- March rows without measured GSC data are excluded rather than silently treated as zero traffic.
- CTR is adjusted by position tier.
- Pages require minimum February evidence (`imp_feb >= 100`).

### What the score means
It is a **decision-support priority score**, not a claim that editing a page will cause recovery or that the baseline explains Google's ranking algorithm.

### Next step
A later model can use the same February information and be judged against this baseline on the same March outcome.